# CDet Reference: interactive tutorial

This notebook walks through using `cdet_reference` to:

1. compute the free Green's function $G_0(i, j; \tau)$ on each lattice
2. run CDet to get perturbative Taylor coefficients in $U$
3. compute the exact interacting Green's function from full diagonalisation
4. extract the single-particle spectral function $A(k, \omega)$

If you've never used the package before, run the cells top to bottom.
Each one is short. The last two cells (ED-based) take 10-30 seconds because
they build and diagonalise a 4096-state matrix.

Before running: install the package with `pip install -e .` from the repo root.

## 1. Free Green's function $G_0(i, j; \tau)$

The lowest-level building block. Built from numerical diagonalisation of the
single-particle Hamiltonian, then summed over eigenstates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cdet_reference.atom.green_function import G0_atom
from cdet_reference.dimer.green_function import G0_2site
from cdet_reference.piflux_square.green_function import G0_4site_piflux
from cdet_reference.hexring.green_function import G0_hexring

# Parameters
beta = 4.0   # inverse temperature
mu   = 0.0   # chemical potential
t    = 1.0   # hopping
tau  = 1.5

# Free Green's function values on each lattice
print(f"atom    G_0(tau)         = {G0_atom(tau, beta, mu):+.10f}")
print(f"dimer   G_0(0,0; tau)    = {G0_2site(0, 0, tau, beta, mu, t):+.10f}")
print(f"dimer   G_0(0,1; tau)    = {G0_2site(0, 1, tau, beta, mu, t):+.10f}")
print(f"piflux  G_0(0,0; tau)    = {G0_4site_piflux(0, 0, tau, beta, mu, t):+.10f}")
print(f"hexring G_0(0,0; tau)    = {G0_hexring(0, 0, tau, beta, mu, t):+.10f}")

## 2. Plot $G_0(i, j; \tau)$ as a function of imaginary time

Each curve is one site pair $(0, j)$ on the hexring. The anti-periodic structure
in $\tau$ is the standard imaginary-time fermion Green's function.

In [ ]:
tau_vals = np.linspace(0.01, beta - 0.01, 100)

plt.figure(figsize=(8, 5))
for j in range(6):
    g_vals = [G0_hexring(0, j, tau, beta, mu, t) for tau in tau_vals]
    plt.plot(tau_vals, g_vals, label=f'G_0(0, {j}; tau)')

plt.xlabel('tau')
plt.ylabel('G_0(0, j; tau)')
plt.title(f'Free Greens function on the 6-site hexring (beta={beta}, mu={mu}, t={t})')
plt.legend(loc='best', fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. CDet Taylor coefficient

Compute the first-order coefficient $c_1(\tau)$ in
$G(\tau; U) = G_0(\tau) + U \cdot c_1(\tau) + U^2 \cdot c_2(\tau) + \ldots$
for the atom, both via CDet integration and via finite-difference of the
exact symbolic answer.

In [ ]:
from scipy.integrate import quad

from cdet_reference.atom.cdet_recursion import C_V
from cdet_reference.atom.green_function import G_exact_atom


def cdet_c1(tau, beta, mu):
    integrand = lambda tau_1: C_V([tau_1], tau_out=tau, tau_in=0.0,
                                   beta=beta, mu=mu)
    val, _ = quad(integrand, 0.0, beta)
    return val


def reference_c1(tau, beta, mu, dU=1e-5):
    g_plus  = G_exact_atom(tau, beta, mu,  dU)
    g_minus = G_exact_atom(tau, beta, mu, -dU)
    return float((g_plus - g_minus) / (2 * dU))


mu_atom = 0.2
print(f"{'tau':>6}  {'c_1 via CDet':>16}  {'c_1 via ref':>14}  {'rel diff':>10}")
print("-" * 55)
for tau in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5]:
    a = cdet_c1(tau, beta, mu_atom)
    b = reference_c1(tau, beta, mu_atom)
    rel = abs(a - b) / max(abs(a), abs(b), 1e-15)
    print(f"{tau:>6.2f}  {a:>+16.10f}  {b:>+14.10f}  {rel:>10.2e}")

## 4. Exact interacting Green's function via ED

For the hexring, the many-body Hilbert space is 4096-dimensional. We build
the full Hamiltonian for arbitrary $U$, diagonalise it, and compute
$G(\tau; U)$ via the Lehmann representation.

This cell is the slowest in the notebook (~15 seconds for the operator
build, then ~10 seconds per $U$ value for the diagonalisation).

In [ ]:
from cdet_reference.hexring.exact_diagonalization import (
    build_hexring_operators, build_H_hexring, G_exact_hexring,
)

print("Building 4096-dim many-body operators...")
ops = build_hexring_operators()
print("done.")

i, j, tau = 0, 1, 1.5
beta_h = 4.0
print(f"\nG(i={i}, j={j}; tau={tau}) on the hexring  (beta={beta_h}, mu=0, t=1)")
print(f"{'U':>6}  {'G_exact(U)':>14}  {'G_0 (free)':>14}  {'difference':>12}")

g_free = G0_hexring(i, j, tau, beta_h, 0.0, 1.0)
for U in [0.0, 0.5, 1.0, 2.0]:
    H = build_H_hexring(beta_h, 0.0, 1.0, U, ops=ops).toarray()
    eigvals, eigvecs = np.linalg.eigh(H)
    g_exact = G_exact_hexring(i, j, tau, beta_h, 0.0, 1.0, U,
                               spin='up', ops=ops,
                               eigh_cache=(eigvals, eigvecs))
    diff = g_exact - g_free
    print(f"{U:>6.2f}  {g_exact:>+14.10f}  {g_free:>+14.10f}  {diff:>+12.4e}")

## 5. Spectral function $A(k, \omega)$

The single-particle spectral function tells us how electrons propagate. We
build it from the Lehmann representation using all 4096 eigenstates.
This is the key transport observable.

In [ ]:
from cdet_reference.spectral_function.spectral_function import spectral_function_exact

beta_s = 8.0
U_s    = 1.0
eta    = 0.05  # Lorentzian broadening

print(f"Diagonalising H at U={U_s}...")
H = build_H_hexring(beta_s, 0.0, 1.0, U_s, ops=ops).toarray()
eigvals, eigvecs = np.linalg.eigh(H)

omega = np.linspace(-4, 4, 401)

plt.figure(figsize=(9, 5))
labels = {0: 'Gamma (k=0)', 1: 'K (k=1)', 2: 'K-prime (k=2)', 3: 'M (k=3)'}
for k in [0, 1, 2, 3]:
    A = spectral_function_exact(k, omega, beta_s, 0.0, 1.0, U_s,
                                 spin='up', eta=eta,
                                 eigh_cache=(eigvals, eigvecs), ops=ops)
    plt.plot(omega, A, label=labels[k], linewidth=1.5)

plt.axvline(0, color='gray', linewidth=0.5, linestyle='--')
plt.xlabel('omega / t')
plt.ylabel('A(k, omega)')
plt.title(f'Spectral function on hexring (beta={beta_s}, U={U_s})')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Next steps

To go deeper:

- The `tests/` directory has the full audit suite: run any
  `tests/test_*_cdet.py` from the repo root to see the verification ladder.
- The `examples/` directory has 4 standalone scripts that produce
  publication-quality outputs.
- See `cdet_reference/<lattice>/derivation.md` for the math behind each
  lattice's free Green's function construction.
- See the top-level `README.md` for the API surface and design overview.